# Data Preprocessing Strategy
**Goal:** Extract domain invariant features, construct robust ground truth (Health Index & RUL labels), apply sequence sliding windows, and prepare the final tensor dataframes for Deep Learning evaluation flow.
*   **Step 1:** Feature Extraction (Maximum FFT bandwidth capped at 1280 Hz to align dimensions with source domain).
*   **Step 2:** Construct Ground Truth Targets. Employs exact CUSUM logic (`calculate_series_health_index`) where $HI$ operates on a pristine $[1.0 \rightarrow 0.0]$ boundary.
*   **Step 3:** Sliding Window Mapping & Tensor Compilation. Generate metadata for lifespan reference.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import random
from typing import List, Dict, Tuple
from tqdm import tqdm

# Add local source to path for imports
sys.path.append("../src")
from CrossDomainFeatureExtractor import CrossDomainFeatureExtractor
from ConstructGroundTruthUsingCUSUM import UnivariateCUSUMDetector

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_PATH = r"../Processed_Data/Downsampled"
OUTPUT_PATH = r"../Processed_Data/LSTM_Inputs"
os.makedirs(OUTPUT_PATH, exist_ok=True)

IS_VAL_SET = False  # Toggle for train vs validation/test behavior. IS_VAL_SET = True forces saving separated windows to prevent timeline jumps
WINDOW_SIZES = [30, 40, 50, 70] # Generate multiple window sizes for both train and val/test

XJTU_SAMPLING_RATE = 25600.0
MAX_FREQ_HZ = 1280.0 # Domain constraint to match LAB Dataset

print(f"Input Path: {os.path.abspath(INPUT_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")
print(f"Is Val Set: {IS_VAL_SET}")
print(f"Window Sizes: {WINDOW_SIZES}")

In [ ]:
def calculate_series_health_index(current_time: float, turning_point_time: float, total_lifespan: float) -> float:
    """
    Calculate the health index based on current time, turning point (Tcp), and total lifespan (Tf).
    Returns a health index value between 1.0 (Healthy) and 0.0 (Failed).
    """
    if total_lifespan == turning_point_time:
        return 0.0  # Prevent division by zero
    return 1.0 - (max((current_time - turning_point_time), 0.0) / (total_lifespan - turning_point_time))

def create_sliding_windows(data_df: pd.DataFrame, window_size: int) -> pd.DataFrame:
    """Transforms continuous features into sliding window sequences (3D -> 2D flattened if needed)."""
    if len(data_df) <= window_size:
        return pd.DataFrame()

    windows = []
    # Exclude metadata and target-related columns from the raw feature sequence
    features_col = [c for c in data_df.columns if c not in ["Change_Point", "Health_Index", "Target_RUL", "Minute", "Bearing_ID", "Status"]]
    
    # Iterate through row sequence to make windows
    for i in range(len(data_df) - window_size):
        window_segment = data_df.iloc[i : i + window_size]
        
        # Flattened Window (e.g. td_rms_t0, td_rms_t1... )
        row_record = {}
        for feat in features_col:
            for t in range(window_size):
                row_record[f"{feat}_t{t}"] = window_segment.iloc[t][feat]
                
        # The label for prediction corresponds to the LAST timestamp in the window sequence
        row_record['Health_Index'] = window_segment.iloc[-1]['Health_Index']
        row_record['Target_RUL'] = window_segment.iloc[-1]['Target_RUL']
        row_record['Bearing_ID'] = window_segment.iloc[-1]['Bearing_ID']
        row_record['Original_Minute'] = window_segment.iloc[-1]['Minute']
        windows.append(row_record)
        
    return pd.DataFrame(windows)

def extract_features_and_build_gt(file_path: str, bearing_id: str) -> Tuple[pd.DataFrame, Dict]:
    """1. Extracts Features per minute. 2. Constructs Ground Truth using CUSUM."""
    df_raw = pd.read_pickle(file_path)
    
    extractor = CrossDomainFeatureExtractor(sampling_rate=XJTU_SAMPLING_RATE, max_freq_hz=MAX_FREQ_HZ)
    features_list = []
    
    for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc=f"Ext. {bearing_id}", leave=False):
        try:
            h_sig = np.array(row['H_Signal'], dtype=float)
        except:
            h_sig = np.zeros(32768) # Fallback if empty array encountered
            
        feats = extractor.extract_all_features(h_sig)
        feats['Minute'] = row['Minute']
        feats['Bearing_ID'] = bearing_id
        features_list.append(feats)
        
    df_features = pd.DataFrame(features_list)
    
    # Eliminate Inf that causes Tensor explosion in Deep Learning due to division-by-zero on variance/std.
    # Convert them to 0.0 before creating windows.
    numeric_cols = df_features.select_dtypes(include=[np.number]).columns
    df_features[numeric_cols] = np.nan_to_num(df_features[numeric_cols].values, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Run CUSUM on RMS to detect degradation start point
    detector = UnivariateCUSUMDetector(baseline_ratio=0.1, k_factor=0.5, h_factor=5.0)
    change_point, _ = detector.fit_predict(df_features['td_rms'].values)
    
    total_life = len(df_features)
    health_indices = []
    rul_labels = []
    
    for min_idx in range(total_life):
        minute = min_idx + 1
        # Calculate standardized Health Index [1.0 -> 0.0 framework]
        hi = calculate_series_health_index(minute, change_point, total_life)
        health_indices.append(hi)
        # Preserve absolute RUL for evaluation reference (in minutes)
        rul_labels.append(total_life - minute)
            
    df_features['Change_Point'] = change_point
    df_features['Health_Index'] = health_indices
    df_features['Target_RUL'] = rul_labels
    
    # Construct metadata dictionary for downstream evaluation routines
    metadata = {
        "Bearing_ID": bearing_id,
        "Total_Lifespan": total_life,
        "Change_Point": change_point
    }
    
    return df_features, metadata

def preprocess_and_slide():
    if not os.path.exists(INPUT_PATH):
        print(f"Directory {INPUT_PATH} not found.")
        return
        
    all_bearings_features = {}
    all_metadata = []
    
    # Search for downsampled pickle files
    downsampled_files = [f for f in os.listdir(INPUT_PATH) if f.endswith(".pkl")]
    if not downsampled_files:
        for cond_folder in os.listdir(INPUT_PATH):
            cf = os.path.join(INPUT_PATH, cond_folder)
            if os.path.isdir(cf):
                downsampled_files.extend([os.path.join(cond_folder, f) for f in os.listdir(cf) if f.endswith(".pkl")])
                
    if not downsampled_files:
        print("No Pickle Files found. Process aborted. Make sure to run Notebook 1 first!")
        return
    
    # Step 1 & 2: Feature Extraction and Ground Truth
    print("--- Step 1 & 2: Feature Extraction & Ground Truth Construction ---")
    for file in tqdm(downsampled_files, desc=f"Bearings Processing Phase"):
        bearing_id = os.path.basename(file).replace("_downsampled.pkl", "")
        file_path = os.path.join(INPUT_PATH, file)
        
        df_feat, metadata = extract_features_and_build_gt(file_path, bearing_id)
        all_bearings_features[bearing_id] = df_feat
        all_metadata.append(metadata)

    # Export Metadata Reference File required during Training/Val evaluation
    metadata_df = pd.DataFrame(all_metadata)
    metadata_path = os.path.join(OUTPUT_PATH, "bearing_metadata.csv")
    metadata_df.to_csv(metadata_path, index=False)
    print(f"Exported Metadata File: {metadata_path}")

    # Step 3: Sliding Windows Process
    print("\n--- Step 3: Sliding Windows & Output Compilation ---")
    for w_size in WINDOW_SIZES:
        if not IS_VAL_SET:
            # Training configuration: Merge across bearings and Random Shuffle (Cross-Domain mix)
            combined_w = []
            for b_id, df_feat in all_bearings_features.items():
                w_df = create_sliding_windows(df_feat, w_size)
                if not w_df.empty:
                    combined_w.append(w_df)
                else:
                    print(f"[Warning] Skipped {b_id} for window size {w_size} (Insufficient length: {len(df_feat)})")
                
            if combined_w:
                final_train_w = pd.concat(combined_w, ignore_index=True)
                final_train_w = final_train_w.sample(frac=1, random_state=42).reset_index(drop=True)
                
                out_file = os.path.join(OUTPUT_PATH, f"processed_train_w{w_size}.csv")
                final_train_w.to_csv(out_file, index=False)
                print(f"Exported (Merged Training Data): {out_file} (Shape Blueprint: {final_train_w.shape})")
            else:
                print(f"[Error] No valid training instances produced for window size {w_size}.")
        else:
            # Validation/Test configuration: Isolate by Bearing_ID, strictly NO SHUFFLING
            for b_id, df_feat in all_bearings_features.items():
                w_df = create_sliding_windows(df_feat, w_size)
                
                if w_df.empty:
                    print(f"[Warning] Skipped {b_id} for window size {w_size} (Insufficient length: {len(df_feat)})")
                    continue
                    
                out_file = os.path.join(OUTPUT_PATH, f"processed_val_{b_id}_w{w_size}.csv")
                w_df.to_csv(out_file, index=False)
                print(f"Exported (Isolated Validation Data): {out_file} (Shape Blueprint: {w_df.shape})")

# Execute Pipeline Process
preprocess_and_slide()